In [74]:
from InputData import *

instance_filename = "AnzahlAuftraege_NEW_10/Construction_a10_o107_m5_an57_ar12.json"
#instance_filename = "AnzahlAuftraege_NEW_50/Construction_a50_o625_m30_an258_ar61.json"
#instance_filename = "Construction_a1_o12_m3_an5_ar3_reduced.json"

data = InputData(instance_filename)


In [76]:
# 2a. Sets
M = list()
W_m = dict()
N_m = dict()
for machine in data.machines:
    M.append(machine.id)
    W_m[machine.id] = machine.default_drivers
    N_m[machine.id] = list()
    for orderItem in data.order_items:
        if orderItem.machine_type == machine.type:
            N_m[machine.id].append(orderItem.id)

W = list()
for worker in data.workers:
    W.append(worker.personal_number)

A = list()
A_Class = list()
for attachment in data.attachments:
    A.append(attachment.id)
    A_Class.append(attachment.type)
    
N = list()
for orderItem in data.order_items:
    N.append(orderItem.id)

C = list()
N_c = dict()
for order in data.orders:
    C.append(order.site_number)
    N_c[order.site_number] = order.order_item_ids


start_date = data.start_date
end_date = data.end_date


O_t = dict()  # Original mit Startzeiten --> im Modell notwendig

O_t_end = dict()  # Endzeiten
O_t_start_inverted = dict()  # Umgekehrtes O_t (Startzeiten)
O_t_end_inverted = dict()  # Umgekehrtes O_t_end (Endzeiten)

SECONDS_IN_A_DAY = 86400

for orderItem in data.order_items:
    orderID = orderItem.id 

    # Startzeit
    orderItem_start_date = orderItem.start_time
    delta_start = (orderItem_start_date - start_date)
    t_start = delta_start.total_seconds() / SECONDS_IN_A_DAY
    
    # Original O_t: Gruppiert nach Startzeit
    if t_start not in O_t:
        O_t[t_start] = []
    O_t[t_start].append(orderID)
    
    # Invertiertes Dictionary O_t_start_inverted
    O_t_start_inverted[orderID] = t_start

    # Endzeit
    orderItem_end_date = orderItem.end_time
    delta_end = (orderItem_end_date - start_date)
    t_end = delta_end.total_seconds() / SECONDS_IN_A_DAY

    # O_t_end: Gruppiert nach Endzeit
    if t_end not in O_t_end:
        O_t_end[t_end] = []
    O_t_end[t_end].append(orderID)
    
    # Invertiertes Dictionary O_t_end_inverted
    O_t_end_inverted[orderID] = t_end





P_mn = dict()
S_mn = dict()

for m in M:
    for n in N_m[m]:
        P_mn[m,n] = list()
        S_mn[m,n] = list()
        for i in N_m[m]:
            if n != i:
                start_time_n = O_t_start_inverted[n]
                end_time_n = O_t_end_inverted[n]
                start_time_i = O_t_start_inverted[i]
                end_time_i = O_t_end_inverted[i]

                if start_time_n > end_time_i:


                    P_mn[m,n].append(i)

                if start_time_i > end_time_n:


                    S_mn[m,n].append(i)



# dict N_m übersichtlich darstellen
for m in M:
    print("Machine: ", m)
    print("Complete Order Items: ", N_m[m])

    for n in N_m[m]:
        # wenn es nicht existiert, dann ist es None
        print(" Order Item: ", n)
        if (m,n) in P_mn:
            print("  Precedence: ", P_mn[m,n])
        if (m,n) in S_mn:
            print("  Succession: ", S_mn[m,n])
    print("\n")

Machine:  0
Complete Order Items:  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106]
 Order Item:  0
  Precedence:  [19, 42, 43, 44, 45, 46]
  Succession:  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 18, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106]
 Order Item:  1
  Precedence:  [0, 17, 19, 20, 21, 42, 43, 44, 45, 46, 47, 48]
  Successi